In [58]:
import pandas as pd
import json

In [59]:
happiness_df = pd.read_csv("happiness.csv")
steps_df = pd.read_csv("world_map_steps_average.csv")
wine_df = pd.read_csv("wine-consumption-per-capita.csv")
diet_df = pd.read_csv("world_diets.csv")
life_expectancy_df = pd.read_csv("life-expectancy.csv")

### Individual features

##### 1. Happiness (no ISO code)

In [60]:
happiness_df = happiness_df.rename(columns={"Life evaluation (3-year average)": "Life evaluation", "Country name": "country"})


##### 2. Steps (no ISO code)

In [61]:
steps_df = steps_df.rename(columns={"region": "country"})

##### 3. Wine consumption

In [62]:
wine_df = wine_df.rename(
    columns={
        "Alcohol, recorded per capita (15+) consumption (in litres of pure alcohol) - Beverage types: Wine": "Wine Consumption", 
        "Code": "ISO",
        "Entity": "country"
        }
    )


##### 4. Diet

In [63]:
diet_df = diet_df.rename(columns={"Entity": "country", "Code": "ISO"})

In [64]:
#https://www.who.int/news-room/fact-sheets/detail/healthy-diet

def rule80(calories):
    if calories < 1200:
        return calories / 1300 #not enough calories that's dangerous       
    elif calories <= 2100: # recommended calories intake
        return 1.0                    
    else:
        return max(0, 1 - (calories - 2100) / 2600)  #excessive calories intake


diet_df["rule80_score"] = diet_df["Calories intake"].apply(rule80)


##### 5. Life expectancy

In [65]:
life_expectancy_df = life_expectancy_df.rename(columns={"Entity": "country", "Code": "ISO"})


In [66]:
YEARS = list(range(1965, 2025))

h_y    = happiness_df[happiness_df["Year"].isin(YEARS)][["Year", "country", "Life evaluation"]].copy()
le_y   = life_expectancy_df[life_expectancy_df["Year"].isin(YEARS)][["Year", "country", "ISO", "Life expectancy"]].copy()
diet_y = diet_df[diet_df["Year"].isin(YEARS)][["Year", "country", "ISO", "plant_based_ratio", "rule80_score"]].copy()
wine_y = wine_df[wine_df["Year"].isin(YEARS)][["Year", "country", "ISO", "Wine Consumption"]].copy()


### Blue Zone Index

In [67]:
all_countries = le_y[["country", "ISO"]].drop_duplicates()
base = (
    pd.MultiIndex.from_product([all_countries["country"].tolist(), YEARS], names=["country", "Year"])
    .to_frame(index=False)
    .merge(all_countries, on="country", how="left")
)

merged = (
    base
    .merge(le_y[["Year", "ISO", "Life expectancy"]],             on=["Year", "ISO"],     how="left")
    .merge(diet_y[["Year", "ISO", "plant_based_ratio","rule80_score"]], on=["Year", "ISO"],   how="left")
    .merge(wine_y[["Year", "ISO", "Wine Consumption"]],           on=["Year", "ISO"],     how="left")
    .merge(steps_df[["country", "steps_mean_filtered"]],          on="country",           how="left")
    .merge(h_y[["Year", "country", "Life evaluation"]],           on=["Year", "country"], how="left")
)

In [68]:
#we fill the missing data with the closest available data for this country in the closest year were data is available 
merged = merged.sort_values(["country", "Year"])
for col in ["Life expectancy", "plant_based_ratio", "rule80_score", "Wine Consumption", "Life evaluation"]:
    merged[col] = merged.groupby("country")[col].ffill()
    merged[col] = merged.groupby("country")[col].bfill()


In [69]:
merged["plant_based_ratio"]   = merged["plant_based_ratio"]   / merged["plant_based_ratio"].max()
merged["rule80_score"]   = merged["rule80_score"]   / merged["rule80_score"].max()
merged["Wine Consumption"]    = merged["Wine Consumption"]    / merged["Wine Consumption"].max()
merged["steps_mean_filtered"] = merged["steps_mean_filtered"] / merged["steps_mean_filtered"].max()
merged["Life evaluation"]     = merged["Life evaluation"]     / merged["Life evaluation"].max()
merged["blue_zone_index"] = (
    merged["Life evaluation"] +
    merged["steps_mean_filtered"] +
    merged["Wine Consumption"] +
    merged["plant_based_ratio"]
) / 4

In [70]:
merged = merged.fillna(0)

MAP_COLS     = ["blue_zone_index", "Life evaluation", "steps_mean_filtered",
                "Wine Consumption", "plant_based_ratio", "Life expectancy","rule80_score"]

map_out     = {}
scatter_out = {}


In [71]:
for year in YEARS:
    ydf = merged[merged["Year"] == year].copy()
    ydf = ydf.groupby("country")[MAP_COLS].mean().reset_index()
    map_out[str(year)]     = ydf.set_index("country")[MAP_COLS].to_dict(orient="index")
    scatter_out[str(year)] = ydf[["country"] + MAP_COLS].to_dict(orient="records")

with open("blue-zone-index-by-year.json", "w") as f:
    json.dump(map_out, f)

with open("blue-zone-index-scatter-plot-by-year.json", "w") as f:
    json.dump(scatter_out, f)
